## 1. Dataset A – Annotation Inspection

In [22]:
import pandas as pd

# Load and inspect the existing annotation checkpoint without modifying it
annotation_df = pd.read_csv("../data/interim/multilingual_annotation_progress.csv")

print("Shape:", annotation_df.shape)
print("Columns:", annotation_df.columns.tolist())
display(annotation_df.head())

Shape: (29137, 13)
Columns: ['instruction', 'issueType', 'description', 'location', 'priority', 'department', 'urgency', 'detectedLanguage', 'aiConfidence', 'sentiment', 'keywords', 'suggestedResponse', 'manual_category']


,instruction,issueType,description,location,priority,department,urgency,detectedLanguage,aiConfidence,sentiment,keywords,suggestedResponse,manual_category
0,"Classify this complaint: ""Sir, we are facing l...",Water Leakage,"Sir, we are facing lot of water problem in 8th...",Opp to seed of truth church,HIGH,Water Supply,high,English,0.9,frustrated,[],Your complaint has been registered with the Wa...,NaN
1,"Classify this complaint: ""Doddabommasandra Arc...",Garbage Overflow,"Doddabommasandra Arch se Nanjappa Circle, Nanj...",Location not specified,MEDIUM,Sanitation,medium,Hinglish,0.9,neutral,[],Your complaint has been registered with the Sa...,NaN
2,"Classify this complaint: ""Kachra kachra kachra""",Garbage Overflow,Kachra kachra kachra,Sainik Public School Road Dead end 6th Cross...,MEDIUM,Sanitation,medium,Hinglish,0.9,neutral,[],Your complaint has been registered with the Sa...,NaN
3,"Classify this complaint: ""Garbage dumped next ...",Garbage Overflow,Garbage dumped next to mysore circle bus stop....,Mysore circle bus stop way to KR Market,HIGH,Sanitation,high,English,0.9,frustrated,[],Your complaint has been registered with the Sa...,NaN
4,"Classify this complaint: ""Too many pot holes n...",Pothole,Too many pot holes near GIIS school and petrol...,Near Petrol bunk,HIGH,Road Maintenance,high,English,0.9,frustrated,[],Your complaint has been registered with the Ro...,NaN


In [23]:
# Check annotation progress and the distribution of manual labels
print("Total records:", len(annotation_df))
print("\nManual annotation status:")
print(annotation_df["manual_category"].value_counts(dropna=False))

print("\nNumber of labelled records:",
      annotation_df["manual_category"].notna().sum())

print("Number of unlabelled records:",
      annotation_df["manual_category"].isna().sum())

Total records: 29137

Manual annotation status:
manual_category
NaN                                      27757
Road Infrastructure                        350
Water Supply                               252
Waste Management                           188
Exclude                                    156
Traffic and Transport                      122
Drainage and Sewerage                      102
Street Lighting                             71
Civic Infrastructure and Encroachment       66
Public Sanitation                           25
Environment and Public Spaces               22
Electricity and Utilities                   13
Public Safety                               10
Animal Management                            3
Name: count, dtype: int64

Number of labelled records: 1380
Number of unlabelled records: 27757


In [24]:
# Separate manually labelled, excluded and unlabelled complaints
labelled_df = annotation_df[
    annotation_df["manual_category"].notna() &
    (annotation_df["manual_category"] != "Exclude")
].copy()

excluded_df = annotation_df[
    annotation_df["manual_category"] == "Exclude"
].copy()

unlabelled_df = annotation_df[
    annotation_df["manual_category"].isna()
].copy()

# Verify the separation
print("Labelled:", len(labelled_df))
print("Excluded:", len(excluded_df))
print("Unlabelled:", len(unlabelled_df))

print("\nTarget category distribution:")
print(labelled_df["manual_category"].value_counts())

Labelled: 1224
Excluded: 156
Unlabelled: 27757

Target category distribution:
manual_category
Road Infrastructure                      350
Water Supply                             252
Waste Management                         188
Traffic and Transport                    122
Drainage and Sewerage                    102
Street Lighting                           71
Civic Infrastructure and Encroachment     66
Public Sanitation                         25
Environment and Public Spaces             22
Electricity and Utilities                 13
Public Safety                             10
Animal Management                          3
Name: count, dtype: int64


In [25]:
# Compare complaint content without considering the manual label
content_cols = [
    col for col in annotation_df.columns
    if col != "manual_category"
]

# Find identical complaint records
duplicate_mask = annotation_df.duplicated(
    subset=content_cols,
    keep=False
)

duplicate_records = annotation_df.loc[duplicate_mask].copy()

print("Total records involved in exact duplicates:",
      len(duplicate_records))

# Identify duplicate groups containing multiple annotation statuses
duplicate_groups = (
    duplicate_records
    .groupby(content_cols, dropna=False)["manual_category"]
    .agg(
        record_count="size",
        labels=lambda x: list(x.dropna().unique()),
        labelled_count=lambda x: x.notna().sum()
    )
)

print("\nDuplicate groups:", len(duplicate_groups))

print("\nDuplicate groups with labelled and unlabelled records:")
print(
    duplicate_groups[
        (duplicate_groups["labelled_count"] > 0) &
        (duplicate_groups["labelled_count"] < duplicate_groups["record_count"])
    ].head(10)
)

Total records involved in exact duplicates: 930

Duplicate groups: 271

Duplicate groups with labelled and unlabelled records:
                                                                                                                                                                                                                                                                                                   record_count  \
instruction                                        issueType              description                                        location               priority department             urgency  detectedLanguage aiConfidence sentiment  keywords suggestedResponse                                                  
Classify this complaint: "Cutting down more the... Damaged Infrastructure Cutting down more then100year old tree 8th cros... Location not specified LOW      General Administration low      English          0.9          neutral    []       Your complaint has b

In [26]:
# Identify exact complaint records shared by labelled and unlabelled datasets
labelled_keys = set(
    map(tuple, labelled_df[content_cols].fillna("<NA>").values)
)

unlabelled_keys = set(
    map(tuple, unlabelled_df[content_cols].fillna("<NA>").values)
)

overlap_keys = labelled_keys.intersection(unlabelled_keys)

print("Exact records shared between labelled and unlabelled:",
      len(overlap_keys))

Exact records shared between labelled and unlabelled: 20


In [27]:
# Remove exact records that already exist in the labelled dataset
labelled_keys = set(
    map(tuple, labelled_df[content_cols].fillna("<NA>").values)
)

unlabelled_keys = unlabelled_df[content_cols].fillna("<NA>").apply(
    tuple, axis=1
)

# Keep only unlabelled records that are not exact copies of labelled records
overlap_mask = unlabelled_keys.isin(labelled_keys)

unlabelled_prediction_df = unlabelled_df.loc[
    ~overlap_mask
].copy()

# Verify the resulting datasets
print("Original labelled records:", len(labelled_df))
print("Original unlabelled records:", len(unlabelled_df))
print("Unlabelled records removed:", overlap_mask.sum())
print("Final prediction pool:", len(unlabelled_prediction_df))

Original labelled records: 1224
Original unlabelled records: 27757
Unlabelled records removed: 71
Final prediction pool: 27686


In [28]:
# Check missing and empty descriptions in both datasets
for name, df in {
    "Labelled": labelled_df,
    "Prediction pool": unlabelled_prediction_df
}.items():

    missing = df["description"].isna().sum()

    empty = (
        df["description"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"\n{name}")
    print("Total records:", len(df))
    print("Missing descriptions:", missing)
    print("Empty descriptions:", empty)

# Inspect the shortest labelled complaints
print("\nShortest labelled descriptions:")
print(
    labelled_df.assign(
        text_length=labelled_df["description"].fillna("").str.strip().str.len()
    )[["description", "manual_category", "text_length"]]
    .sort_values("text_length")
    .head(15)
)


Labelled
Total records: 1224
Missing descriptions: 0
Empty descriptions: 0

Prediction pool
Total records: 27686
Missing descriptions: 0
Empty descriptions: 0

Shortest labelled descriptions:
            description        manual_category  text_length
13726        नया शौचालय      Public Sanitation           10
14760        बड़ा गड्ढा    Road Infrastructure           10
21514        खराब सड़क।    Road Infrastructure           10
15585       bada gaddha    Road Infrastructure           11
3817       कचरा संग्रहण       Waste Management           12
4302       कचरा साफ करो       Waste Management           12
334        सड़क पर कचरा       Waste Management           12
8166      Street lights        Street Lighting           13
5728      सड़क पर गड्ढे    Road Infrastructure           13
14583    सिग्नल उल्लंघन  Traffic and Transport           14
27133   Paani ka risaav           Water Supply           15
19899   Broken footpath    Road Infrastructure           15
23665   शौचालय नहीं हैं    

In [29]:
# Find descriptions associated with more than one manual category
description_label_counts = (
    labelled_df.groupby("description")["manual_category"]
    .nunique()
)

conflicting_descriptions = description_label_counts[
    description_label_counts > 1
]

print("Unique descriptions:", labelled_df["description"].nunique())
print("Repeated descriptions:", labelled_df["description"].duplicated().sum())
print("Descriptions with conflicting categories:",
      len(conflicting_descriptions))

# Display conflicting examples for manual inspection
if len(conflicting_descriptions) > 0:
    conflict_examples = labelled_df[
        labelled_df["description"].isin(conflicting_descriptions.index)
    ][["description", "manual_category"]].sort_values("description")

    print("\nConflicting examples:")
    print(conflict_examples.to_string(index=False))
    

Unique descriptions: 1223
Repeated descriptions: 1
Descriptions with conflicting categories: 0


In [30]:
# Check the detected language distribution in labelled data
print("Overall language distribution:")
print(labelled_df["detectedLanguage"].value_counts(dropna=False))

# Check language distribution within each target category
language_category = pd.crosstab(
    labelled_df["manual_category"],
    labelled_df["detectedLanguage"]
)

print("\nLanguage distribution by category:")
print(language_category)

# Display percentages for easier comparison
print("\nLanguage percentages by category:")
print(
    pd.crosstab(
        labelled_df["manual_category"],
        labelled_df["detectedLanguage"],
        normalize="index"
    ).mul(100).round(2)
)

Overall language distribution:
detectedLanguage
Hinglish    412
Hindi       409
English     403
Name: count, dtype: int64

Language distribution by category:
detectedLanguage                       English  Hindi  Hinglish
manual_category                                                
Animal Management                            2      0         1
Civic Infrastructure and Encroachment       18     21        27
Drainage and Sewerage                       39     34        29
Electricity and Utilities                    7      4         2
Environment and Public Spaces                6      8         8
Public Safety                                3      2         5
Public Sanitation                            7     10         8
Road Infrastructure                        115    112       123
Street Lighting                             17     25        29
Traffic and Transport                       44     34        44
Waste Management                            59     73        56
Water Supp

## 2. Dataset B – Category Verification

In [31]:
# Load Dataset B using the exact path from the notebook
ichangemycity_df = pd.read_csv(
    "../data/interim/civic_complaints_labelled.csv"
)

# Verify the actual dataset
print("Dataset B shape:", ichangemycity_df.shape)
print("Columns:", ichangemycity_df.columns.tolist())

# Compare the target categories in both datasets
categories_A = set(labelled_df["manual_category"].dropna().unique())
categories_B = set(ichangemycity_df["target_category"].dropna().unique())

print("\nCategories in Dataset A:", len(categories_A))
print("Categories in Dataset B:", len(categories_B))

print("\nMissing from Dataset B:")
print(sorted(categories_A - categories_B))

print("\nExtra in Dataset B:")
print(sorted(categories_B - categories_A))

print("\nDo both datasets have exactly the same categories?",
      categories_A == categories_B)

# Compare category-wise record counts
print("\nDataset A distribution:")
print(labelled_df["manual_category"].value_counts().sort_index())

print("\nDataset B distribution:")
print(ichangemycity_df["target_category"].value_counts().sort_index())

Dataset B shape: (15889, 18)
Columns: ['created_at', 'ward_id', 'title', 'description', 'sub_category_id', 'civic_agency_id', 'location', 'address', 'latitude', 'longitude', 'ward_title', 'category_id', 'category_title', 'sub_category_title', 'civic_agency_title', 'complaint_status_title', 'comment_count', 'target_category']

Categories in Dataset A: 12
Categories in Dataset B: 12

Missing from Dataset B:
[]

Extra in Dataset B:
[]

Do both datasets have exactly the same categories? True

Dataset A distribution:
manual_category
Animal Management                          3
Civic Infrastructure and Encroachment     66
Drainage and Sewerage                    102
Electricity and Utilities                 13
Environment and Public Spaces             22
Public Safety                             10
Public Sanitation                         25
Road Infrastructure                      350
Street Lighting                           71
Traffic and Transport                    122
Waste Management

## 3. Dataset Compatibility and Text Preparation

In [32]:
import re

# Create independent copies for inspection
dataset_a_check = labelled_df.copy()
dataset_b_check = ichangemycity_df.copy()

# Construct the complaint text for Dataset B
dataset_b_check["model_text"] = (
    dataset_b_check["title"].fillna("").astype(str).str.strip()
    + " "
    + dataset_b_check["description"].fillna("").astype(str).str.strip()
).str.strip()

# Use the description directly for Dataset A
dataset_a_check["model_text"] = (
    dataset_a_check["description"].fillna("").astype(str).str.strip()
)

# Check missing or empty complaint text
print("Dataset A empty texts:",
      dataset_a_check["model_text"].eq("").sum())

print("Dataset B empty texts:",
      dataset_b_check["model_text"].eq("").sum())

# Normalize text for duplicate comparison
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

dataset_a_check["normalized_text"] = (
    dataset_a_check["model_text"].apply(normalize_text)
)

dataset_b_check["normalized_text"] = (
    dataset_b_check["model_text"].apply(normalize_text)
)

# Find exact normalized text overlap between datasets
overlap_texts = (
    set(dataset_a_check["normalized_text"])
    .intersection(set(dataset_b_check["normalized_text"]))
)

print("\nUnique overlapping complaints:", len(overlap_texts))

# Display a few overlapping examples
print("\nOverlapping examples:")

display(
    dataset_a_check[
        dataset_a_check["normalized_text"].isin(overlap_texts)
    ][["model_text", "manual_category"]].head(10)
)

Dataset A empty texts: 0
Dataset B empty texts: 0

Unique overlapping complaints: 0

Overlapping examples:


,model_text,manual_category


## 4. Combined Dataset Inspection

In [33]:
# Prepare Dataset A: multilingual complaints
dataset_a = labelled_df[["description", "manual_category"]].copy()

dataset_a["model_text"] = (
    dataset_a["description"].fillna("").astype(str).str.strip()
)

dataset_a["target_category"] = dataset_a["manual_category"]
dataset_a["source_dataset"] = "Dataset_A_Multilingual"

dataset_a = dataset_a[
    ["model_text", "target_category", "source_dataset"]
]


# Prepare Dataset B: English complaints
dataset_b = ichangemycity_df[["title", "description", "target_category"]].copy()

dataset_b["model_text"] = (
    dataset_b["title"].fillna("").astype(str).str.strip()
    + " "
    + dataset_b["description"].fillna("").astype(str).str.strip()
).str.strip()

dataset_b["source_dataset"] = "Dataset_B_English"

dataset_b = dataset_b[
    ["model_text", "target_category", "source_dataset"]
]


# Combine both datasets without modifying the original dataframes
combined_df = pd.concat(
    [dataset_a, dataset_b],
    ignore_index=True
)

# Verify the combined dataset
print("Combined dataset shape:", combined_df.shape)

print("\nSource distribution:")
print(combined_df["source_dataset"].value_counts())

print("\nTarget category distribution:")
print(combined_df["target_category"].value_counts().sort_index())

print("\nMissing values:")
print(combined_df.isnull().sum())

display(combined_df.head())

Combined dataset shape: (17113, 3)

Source distribution:
source_dataset
Dataset_B_English         15889
Dataset_A_Multilingual     1224
Name: count, dtype: int64

Target category distribution:
target_category
Animal Management                         889
Civic Infrastructure and Encroachment     242
Drainage and Sewerage                    1026
Electricity and Utilities                 274
Environment and Public Spaces             764
Public Safety                             206
Public Sanitation                         530
Road Infrastructure                      5377
Street Lighting                          1572
Traffic and Transport                    1205
Waste Management                         4425
Water Supply                              603
Name: count, dtype: int64

Missing values:
model_text         0
target_category    0
source_dataset     0
dtype: int64


,model_text,target_category,source_dataset
0,Sadak paar karne wala concrete ka puliya adhur...,Road Infrastructure,Dataset_A_Multilingual
1,"सीएमसी का पानी जारी नहीं किया जा रहा है, वे बा...",Water Supply,Dataset_A_Multilingual
2,Garbage dump all over,Waste Management,Dataset_A_Multilingual
3,Puttenahalli jheel road jo Arekere ki or jaati...,Road Infrastructure,Dataset_A_Multilingual
4,Desilted Mud is not cleared making the road di...,Waste Management,Dataset_A_Multilingual


## 5. Duplicate and Conflict Resolution

In [34]:
# Normalize complaint text for duplicate detection
combined_df["normalized_text"] = (
    combined_df["model_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Count repeated complaint descriptions
print("Total records:", len(combined_df))
print("Unique complaint texts:", combined_df["normalized_text"].nunique())

print(
    "Repeated complaint records:",
    combined_df["normalized_text"].duplicated().sum()
)

# Check whether repeated texts have conflicting target categories
duplicate_groups = (
    combined_df.groupby("normalized_text")
    .agg(
        record_count=("target_category", "size"),
        category_count=("target_category", "nunique"),
        categories=("target_category", lambda x: list(x.unique())),
        sources=("source_dataset", lambda x: list(x.unique()))
    )
)

conflicts = duplicate_groups[
    duplicate_groups["category_count"] > 1
]

print("\nDuplicate groups:", (duplicate_groups["record_count"] > 1).sum())
print("Conflicting duplicate groups:", len(conflicts))

# Display conflicting examples, if any
if len(conflicts) > 0:
    display(conflicts.head(10))

Total records: 17113
Unique complaint texts: 16525
Repeated complaint records: 588

Duplicate groups: 228
Conflicting duplicate groups: 22


,record_count,category_count,categories,sources
normalized_text,,,,
checking checking the complaints,2,2,"[Animal Management, Waste Management]",[Dataset_B_English]
foothpaths walkability is very poor foothpaths walkability is very poor,2,2,"[Environment and Public Spaces, Waste Management]",[Dataset_B_English]
garbage piling up at roadside garbage piling up at roadside,2,2,"[Road Infrastructure, Waste Management]",[Dataset_B_English]
horrible traffic horrible traffic,3,2,"[Animal Management, Traffic and Transport]",[Dataset_B_English]
loads of sand accumulation near the flyover walls at bellandur re... loads of sand accumulation near the flyover walls at bellandur region...that sand is acting as ground for weeds and plants which can damage the bridges in long run...there is noone looking after to collect the sand accumulation,2,2,"[Waste Management, Road Infrastructure]",[Dataset_B_English]
"loud speaker of hawkers in auto, van hawkers have again started loudly announcing about their wares like plastic vessels, garments, led bulbs etc in the evening in all the main and cross road if vishwapriyanagar, begur road. this is a residential layout, and causes disturbance to senior citizens and children studying. they don't seem to be scared of pink cars of bangalore police",2,2,"[Road Infrastructure, Public Safety]",[Dataset_B_English]
no parking no parking,8,2,"[Waste Management, Traffic and Transport]",[Dataset_B_English]
no street lights no street lights,3,2,"[Street Lighting, Road Infrastructure]",[Dataset_B_English]
parking on footpath parking on footpath,13,2,"[Traffic and Transport, Road Infrastructure]",[Dataset_B_English]


In [35]:
# Identify all complaint texts assigned to multiple categories
conflicting_texts = conflicts.index

# Retrieve every record belonging to those groups
conflict_records = combined_df[
    combined_df["normalized_text"].isin(conflicting_texts)
].copy()

# Display the complete complaint text and its assigned category
for text, group in conflict_records.groupby("normalized_text", sort=False):

    print("\n" + "=" * 100)
    print("COMPLAINT:")
    print(group["model_text"].iloc[0])

    print("\nASSIGNED CATEGORIES:")
    print(group[["target_category", "source_dataset"]].to_string(index=False))

    print("Number of records:", len(group))


COMPLAINT:
Street Light not working Street Light not working

ASSIGNED CATEGORIES:
          target_category    source_dataset
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_English
          Street Lighting Dataset_B_

In [36]:
# Identify all records belonging to conflicting complaint groups
conflicting_texts = conflicts.index

conflict_review_df = combined_df[
    combined_df["normalized_text"].isin(conflicting_texts)
].copy()

# Keep records without conflicting duplicate labels in a separate dataframe
conflict_free_df = combined_df[
    ~combined_df["normalized_text"].isin(conflicting_texts)
].copy()

# Verify the separation
print("Original combined dataset:", len(combined_df))
print("Conflict review records:", len(conflict_review_df))
print("Conflict-free records:", len(conflict_free_df))

print("\nConflicting complaint groups:",
      conflict_review_df["normalized_text"].nunique())

# Check that no conflicting groups remain in the clean dataframe
print("\nRemaining conflicting groups:",
      conflict_free_df.groupby("normalized_text")["target_category"]
      .nunique().gt(1).sum())


Original combined dataset: 17113
Conflict review records: 135
Conflict-free records: 16978

Conflicting complaint groups: 22

Remaining conflicting groups: 0


In [37]:
# Summarize all conflicting complaint groups for review
conflict_summary = (
    conflict_review_df
    .groupby("normalized_text")
    .agg(
        record_count=("target_category", "size"),
        categories=("target_category", lambda x: list(x.unique())),
        category_counts=("target_category", lambda x: x.value_counts().to_dict()),
        complaint=("model_text", "first")
    )
    .reset_index()
)

# Display all 22 groups
for _, row in conflict_summary.iterrows():
    print("\nComplaint:", row["complaint"])
    print("Records:", row["record_count"])
    print("Categories:", row["category_counts"])
    print("-" * 80)


Complaint: Checking Checking the complaints
Records: 2
Categories: {'Animal Management': 1, 'Waste Management': 1}
--------------------------------------------------------------------------------

Complaint: Foothpaths walkability is very poor Foothpaths walkability is very poor
Records: 2
Categories: {'Environment and Public Spaces': 1, 'Waste Management': 1}
--------------------------------------------------------------------------------

Complaint: Garbage piling up at roadside Garbage piling up at roadside
Records: 2
Categories: {'Road Infrastructure': 1, 'Waste Management': 1}
--------------------------------------------------------------------------------

Complaint: Horrible traffic Horrible traffic
Records: 3
Categories: {'Traffic and Transport': 2, 'Animal Management': 1}
--------------------------------------------------------------------------------

Complaint: Loads of sand accumulation near the flyover walls at bellandur re... Loads of sand accumulation near the flyover w

In [38]:
# Create a separate copy for applying the conflict-review decisions
reviewed_df = conflict_review_df.copy()

# Preserve the original labels before making corrections
reviewed_df["original_category"] = reviewed_df["target_category"]

# Store the agreed category decisions using unique text prefixes
review_decisions = {
    "checking checking the complaints": ("Exclude", "Insufficient complaint information"),
    "foothpaths walkability is very poor": ("Road Infrastructure", "Footpath condition"),
    "garbage piling up at roadside": ("Waste Management", "Garbage accumulation"),
    "horrible traffic": ("Traffic and Transport", "Traffic congestion"),
    "loads of sand accumulation near the flyover walls": ("Waste Management", "Accumulated sand requiring removal"),
    "loud speaker of hawkers in auto, van": ("Environment and Public Spaces", "Noise pollution"),
    "no parking no parking": ("Traffic and Transport", "Parking violation"),
    "no street lights no street lights": ("Street Lighting", "Street lighting issue"),
    "parking on footpath": ("Traffic and Transport", "Footpath parking violation"),
    "please fix! please fix!": ("Exclude", "Insufficient complaint information"),
    "please fix!! please fix!!": ("Exclude", "Insufficient complaint information"),
    "please fix!!! please fix!!!": ("Exclude", "Insufficient complaint information"),
    "please repair please repair": ("Exclude", "Insufficient complaint information"),
    "repair street ligh": ("Street Lighting", "Street light repair"),
    "riding without helmet": ("Traffic and Transport", "Traffic safety violation"),
    "street light not working": ("Street Lighting", "Street light malfunction"),
    "street lights are not working": ("Street Lighting", "Street light malfunction"),
    "street lights not working": ("Street Lighting", "Street light malfunction"),
    "the neighborhood is filled with illegal slum homes": ("Civic Infrastructure and Encroachment", "Illegal encroachment"),
    "the truth is that the": ("Exclude", "Incomplete complaint"),
    "water leakage": ("Water Supply", "Water leakage"),
    "wedding halls of 2nd main industrial town": ("Traffic and Transport", "Parking causing traffic congestion")
}

# Match each decision to exactly one conflicting complaint group
for prefix, (category, reason) in review_decisions.items():

    matching_texts = conflict_summary.loc[
        conflict_summary["normalized_text"].str.startswith(prefix),
        "normalized_text"
    ]

    if len(matching_texts) != 1:
        raise ValueError(
            f"Expected exactly one matching group for: {prefix}. "
            f"Found: {len(matching_texts)}"
        )

    matched_text = matching_texts.iloc[0]
    mask = reviewed_df["normalized_text"] == matched_text

    # Apply the reviewed category and retain the decision reason
    reviewed_df.loc[mask, "target_category"] = category
    reviewed_df.loc[mask, "review_reason"] = reason

# Mark records that should not be used for supervised training
reviewed_df["review_status"] = reviewed_df["target_category"].apply(
    lambda x: "Excluded" if x == "Exclude" else "Corrected/Retained"
)

# Separate usable reviewed records and excluded records
reviewed_usable_df = reviewed_df[
    reviewed_df["target_category"] != "Exclude"
].copy()

reviewed_excluded_df = reviewed_df[
    reviewed_df["target_category"] == "Exclude"
].copy()

# Verify the results
print("Reviewed records:", len(reviewed_df))
print("Usable reviewed records:", len(reviewed_usable_df))
print("Excluded reviewed records:", len(reviewed_excluded_df))

print("\nReview status:")
print(reviewed_df["review_status"].value_counts())

print("\nUpdated category distribution:")
print(reviewed_usable_df["target_category"].value_counts())

print("\nRemaining conflicting groups:")
print(
    reviewed_usable_df.groupby("normalized_text")["target_category"]
    .nunique().gt(1).sum()
)

Reviewed records: 135
Usable reviewed records: 116
Excluded reviewed records: 19

Review status:
review_status
Corrected/Retained    116
Excluded               19
Name: count, dtype: int64

Updated category distribution:
target_category
Street Lighting                          67
Traffic and Transport                    36
Waste Management                          4
Water Supply                              3
Civic Infrastructure and Encroachment     2
Environment and Public Spaces             2
Road Infrastructure                       2
Name: count, dtype: int64

Remaining conflicting groups:
0


## 6. Final Modelling Dataset Preparation

In [39]:
# Combine conflict-free records with reviewed usable records
modelling_df = pd.concat(
    [conflict_free_df, reviewed_usable_df],
    ignore_index=True
)

# Verify the combined modelling dataset
print("Modelling dataset shape:", modelling_df.shape)

print("\nSource distribution:")
print(modelling_df["source_dataset"].value_counts())

print("\nCategory distribution:")
print(modelling_df["target_category"].value_counts().sort_index())

# Verify that no conflicting labels remain
remaining_conflicts = (
    modelling_df.groupby("normalized_text")["target_category"]
    .nunique()
)

print("\nConflicting complaint groups remaining:",
      (remaining_conflicts > 1).sum())

# Inspect repeated complaint groups that now have consistent labels
repeated_groups = (
    modelling_df.groupby("normalized_text")
    .agg(
        record_count=("target_category", "size"),
        category_count=("target_category", "nunique"),
        category=("target_category", "first")
    )
)

repeated_groups = repeated_groups[
    repeated_groups["record_count"] > 1
]

print("\nRepeated complaint groups:",
      len(repeated_groups))

print("\nRepeated groups with consistent categories:",
      (repeated_groups["category_count"] == 1).sum())

print("\nRepeated groups with conflicting categories:",
      (repeated_groups["category_count"] > 1).sum())

Modelling dataset shape: (17094, 7)

Source distribution:
source_dataset
Dataset_B_English         15870
Dataset_A_Multilingual     1224
Name: count, dtype: int64

Category distribution:
target_category
Animal Management                         887
Civic Infrastructure and Encroachment     244
Drainage and Sewerage                    1021
Electricity and Utilities                 272
Environment and Public Spaces             764
Public Safety                             205
Public Sanitation                         529
Road Infrastructure                      5363
Street Lighting                          1577
Traffic and Transport                    1214
Waste Management                         4414
Water Supply                              604
Name: count, dtype: int64

Conflicting complaint groups remaining: 0

Repeated complaint groups: 222

Repeated groups with consistent categories: 222

Repeated groups with conflicting categories: 0


In [40]:
# Keep one representative of each normalized complaint text
final_dataset = modelling_df.drop_duplicates(
    subset=["normalized_text"],
    keep="first"
).copy()

# Verify the deduplication
print("Original modelling records:", len(modelling_df))
print("Deduplicated records:", len(final_dataset))
print("Records removed:", len(modelling_df) - len(final_dataset))

print("\nSource distribution:")
print(final_dataset["source_dataset"].value_counts())

print("\nFinal category distribution:")
print(final_dataset["target_category"].value_counts().sort_index())

# Verify that no repeated texts or conflicting labels remain
print("\nRepeated complaint texts:",
      final_dataset["normalized_text"].duplicated().sum())

print("Conflicting categories:",
      final_dataset.groupby("normalized_text")["target_category"]
      .nunique().gt(1).sum())

Original modelling records: 17094
Deduplicated records: 16519
Records removed: 575

Source distribution:
source_dataset
Dataset_B_English         15296
Dataset_A_Multilingual     1223
Name: count, dtype: int64

Final category distribution:
target_category
Animal Management                         868
Civic Infrastructure and Encroachment     242
Drainage and Sewerage                    1006
Electricity and Utilities                 269
Environment and Public Spaces             758
Public Safety                             203
Public Sanitation                         481
Road Infrastructure                      5224
Street Lighting                          1472
Traffic and Transport                    1159
Waste Management                         4235
Water Supply                              602
Name: count, dtype: int64

Repeated complaint texts: 0
Conflicting categories: 0


## 7. Validate the saved dataset files

In [42]:
from pathlib import Path
import pandas as pd

# Check whether the two intended output files already exist
supervised_path = Path("../data/interim/civicsense_supervised_dataset.csv")
prediction_path = Path("../data/interim/civicsense_prediction_pool.csv")

for path in [supervised_path, prediction_path]:

    print(f"\nFile: {path}")
    print("Exists:", path.exists())

    # If the file exists, inspect its dimensions and columns
    if path.exists():
        existing_df = pd.read_csv(path)
        print("Shape:", existing_df.shape)
        print("Columns:", existing_df.columns.tolist())


File: ..\data\interim\civicsense_supervised_dataset.csv
Exists: True
Shape: (16519, 3)
Columns: ['model_text', 'target_category', 'source_dataset']

File: ..\data\interim\civicsense_prediction_pool.csv
Exists: True
Shape: (27686, 13)
Columns: ['instruction', 'issueType', 'description', 'location', 'priority', 'department', 'urgency', 'detectedLanguage', 'aiConfidence', 'sentiment', 'keywords', 'suggestedResponse', 'manual_category']


In [43]:
# Load the already-saved datasets for final verification
supervised_check = pd.read_csv(supervised_path)
prediction_check = pd.read_csv(prediction_path)

# Verify the expected dimensions and column structures
assert supervised_check.shape == (16519, 3)
assert prediction_check.shape == (27686, 13)

assert supervised_check.columns.tolist() == [
    "model_text", "target_category", "source_dataset"
]

assert prediction_check.columns.tolist() == [
    "instruction", "issueType", "description", "location",
    "priority", "department", "urgency", "detectedLanguage",
    "aiConfidence", "sentiment", "keywords", "suggestedResponse",
    "manual_category"
]

# Confirm that both saved files passed validation
print("Supervised dataset: VALIDATED")
print("Prediction pool: VALIDATED")
print("Both datasets are ready for the next stage.")

Supervised dataset: VALIDATED
Prediction pool: VALIDATED
Both datasets are ready for the next stage.
